# ⚡ Ultra-Simple 30-Line Pure ML Baseline (Logistic Regression)

**The absolute simplest possible competitive baseline for PSTU DataThon:**
* No complex feature engineering
* No SMOTE or synthetic noise
* 10-Fold Stratified Cross Validation
* Standard scaling + Regularized Logistic Regression with `class_weight='balanced'`
* Runs on CPU in ~15-20 seconds!


In [ ]:
import os, warnings, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
warnings.filterwarnings('ignore')

# 1. Load Data
DATA_DIR = next(d for d in ['/kaggle/input/competitions/pstu-data-thon-2026-vol-1', 'pstu-data-thon-2026-vol-1', '.'] if os.path.exists(os.path.join(d, 'train.csv')))
train, test = pd.read_csv(f'{DATA_DIR}/train.csv'), pd.read_csv(f'{DATA_DIR}/test.csv')
y = train['TARGET'].values
test_ids = test['id'].values if 'id' in test.columns else np.arange(len(test))

# 2. Clean numerical features
num_cols = [c for c in train.columns if c.startswith('feat_') and train[c].dtype != 'object']
X_tr = train[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values
X_te = test[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0).values

# 3. Scale features
scaler = StandardScaler()
X_tr = np.nan_to_num(scaler.fit_transform(X_tr))
X_te = np.nan_to_num(scaler.transform(X_te))

# 4. 10-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))
test_preds = np.zeros(len(X_te))

for tr_idx, va_idx in skf.split(X_tr, y):
    clf = LogisticRegression(C=0.05, penalty='l2', class_weight='balanced', max_iter=500, random_state=42)
    clf.fit(X_tr[tr_idx], y[tr_idx])
    oof_preds[va_idx] = clf.predict_proba(X_tr[va_idx])[:, 1]
    test_preds += clf.predict_proba(X_te)[:, 1] / 10.0

# 5. Find Optimal F1 Threshold
thresholds = np.arange(0.10, 0.90, 0.001)
best_f1, best_t = max([(f1_score(y, (oof_preds >= t).astype(int)), t) for t in thresholds if (oof_preds >= t).sum() > 0])

print(f'OOF ROC-AUC: {roc_auc_score(y, oof_preds):.5f} | Peak F1: {best_f1:.5f} @ t={best_t:.4f}')

# 6. Export submission
out_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'
pd.DataFrame({'id': test_ids, 'TARGET': (test_preds >= best_t).astype(int)}).to_csv(out_path, index=False)
print(f'Done! Saved {out_path}')
